# LSTM 模型

## 1. 什么是 LSTM？
**LSTM（Long Short-Term Memory，长短期记忆网络）** 是一种特殊的 RNN（循环神经网络），专门设计用于解决长序列训练中的**梯度消失**和**梯度爆炸**问题。

LSTM 的核心思想是引入了 **“细胞状态”（Cell State）** 和 **“门控机制”（Gate Mechanism）**，使其能够选择性地记住或遗忘信息。

### LSTM 的三个核心门控：
1. **遗忘门（Forget Gate）**：决定要从细胞状态中丢弃什么信息。（例如：看到新的主语时，遗忘旧主语的性别信息）
2. **输入门（Input Gate）**：决定要向细胞状态中更新什么新信息。
3. **输出门（Output Gate）**：决定基于当前的细胞状态，输出什么值给下一个隐藏状态。

---

## 2. LSTM 与 RNN 的对比

| 特性 | RNN（循环神经网络） | LSTM（长短期记忆网络） |
| :--- | :--- | :--- |
| **结构复杂度** | 结构简单，只有一个 tanh 层 | 结构复杂，包含四个交互层（遗忘门、输入门、候选层、输出门） |
| **记忆能力** | **短期记忆**，难以捕捉长距离依赖 | **长短期记忆**，通过细胞状态贯穿始终，能记住很久之前的信息 |
| **梯度问题** | 容易出现**梯度消失**（Gradient Vanishing）或梯度爆炸 | 通过门控机制有效缓解梯度消失，适合深层网络和长序列 |
| **训练速度** | 参数少，训练快，但收敛效果可能不如 LSTM | 参数多（是 RNN 的 4 倍），训练慢，需要更多数据 |

---

## 3. 为什么 LSTM 比 RNN 强？（LSTM 的优势）
1. **解决长距离依赖问题**：
   - RNN 在处理长句子时（如“我出生在法国......（中间隔了100个词）......所以我说法语”），很难将开头的“法国”和结尾的“法语”联系起来。
   - LSTM 通过细胞状态（Cell State）这一“高速公路”，可以让信息在长序列中无损传递。

2. **精细的信息控制**：
   - RNN 只是简单地将新旧信息叠加（通过 tanh）。
   - LSTM 可以通过门控（Sigmoid 函数）精确控制：记住 80% 的历史，加入 20% 的新信息。

---

## 4. RNN 与 LSTM 各自的独特优点

### RNN 的优点（何时使用 RNN？）：
- **计算开销小**：结构简单，参数少，推理速度快。
- **适合短序列**：对于不需要长期记忆的任务（如简单的关键词提取、短文本分类），RNN 效果往往足够好且效率更高。
- **易于解释**：相比 LSTM 复杂的内部逻辑，RNN 的行为更容易理解和调试。

### LSTM 的优点（何时使用 LSTM？）：
- **适合长序列任务**：如机器翻译（整句翻译）、语音识别（长音频）、股票预测（长周期趋势）。
- **鲁棒性强**：在含有噪声的数据中，LSTM 能更好地过滤掉无关信息（通过遗忘门）。
- **工业界标准**：在 Transformer 出现之前，LSTM 是 NLP 和时间序列领域的绝对霸主，生态完善。

IMDb 数据集下载链接（示例：标题基本信息）
```py
"title.basics.tsv.gz": "https://datasets.imdbws.com/title.basics.tsv.gz",
"title.ratings.tsv.gz": "https://datasets.imdbws.com/title.ratings.tsv.gz",
"name.basics.tsv.gz": "https://datasets.imdbws.com/name.basics.tsv.gz"


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
import numpy as np
import torch.nn as nn
# 读取 title.basics.tsv
df = pd.read_csv('title.basics.tsv', sep='\t', low_memory=False)

# 简单清洗：去掉缺失值，仅保留需要字段
df = df[['tconst', 'primaryTitle', 'titleType', 'startYear', 'genres']].dropna()
df['startYear'] = pd.to_numeric(df['startYear'], errors='coerce')
df = df.dropna(subset=['startYear'])

# 将 genres 做 LabelEncoder，用于后续 embedding
le = LabelEncoder()
df['genres_id'] = le.fit_transform(df['genres'])

# 构建序列数据：按 startYear 排序，取前 1000 条作为示例
df = df.sort_values('startYear').head(1000)
seq_len = 10
data = df['genres_id'].values

# 构造样本：用前 seq_len-1 个预测第 seq_len 个
X, y = [], []
for i in range(len(data) - seq_len):
    X.append(data[i:i+seq_len-1])
    y.append(data[i+seq_len-1])
X, y = np.array(X), np.array(y)

class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.LongTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = SimpleDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# 简单 LSTM 模型
class SimpleLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    def forward(self, x):
        x = self.emb(x)
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # 取最后一个时间步
        return self.fc(out)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleLSTM(vocab_size=len(le.classes_)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 训练 4 个 epoch
for epoch in range(4):
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}')



Epoch 1, Loss: 1.4495
Epoch 2, Loss: 2.3412
Epoch 3, Loss: 1.3045
Epoch 4, Loss: 1.1383
Epoch 5, Loss: 1.4282


In [5]:
# 评估模型
model.eval()
with torch.no_grad():
    total, correct = 0, 0
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        logits = model(bx)
        preds = torch.argmax(logits, dim=1)
        total += by.size(0)
        correct += (preds == by).sum().item()
    print(f'Accuracy: {correct / total:.4f}')

Accuracy: 0.6061


In [ ]:
import torch
import torch.nn as nn

class BiLSTM(nn.Module):
    """
    双层双向 LSTM 模型
    用于序列建模任务，如文本分类、命名实体识别等
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=2, num_classes=2, dropout=0.5):
        """
        初始化双层双向 LSTM 模型

        参数:
            vocab_size (int): 词汇表大小
            embed_dim (int): 词嵌入维度
            hidden_dim (int): LSTM 隐藏层维度
            num_layers (int): LSTM 层数，默认为 2
            num_classes (int): 分类类别数，默认为 2
            dropout (float): dropout 比例，默认为 0.5
        """
        super(BiLSTM, self).__init__()

        # 词嵌入层：将输入的词索引转换为固定维度的向量
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # 双层双向 LSTM：num_layers=2，bidirectional=True 表示双向
        # 输入维度为 embed_dim，隐藏维度为 hidden_dim
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            bidirectional=True, batch_first=True, dropout=dropout)

        # 全连接层：将 LSTM 输出映射到类别数
        # 由于是双向 LSTM，输出维度为 hidden_dim * 2
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

        # Dropout 层：防止过拟合
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        前向传播过程

        参数:
            x (Tensor): 输入的词索引序列，形状为 (batch_size, seq_len)

        返回:
            out (Tensor): 分类 logits，形状为 (batch_size, num_classes)
        """
        # 1. 词嵌入：将词索引转换为向量
        # 输出形状：(batch_size, seq_len, embed_dim)
        embeds = self.embedding(x)

        # 2. 通过双层双向 LSTM
        # lstm_out: (batch_size, seq_len, hidden_dim * 2)
        # hidden: (num_layers * 2, batch_size, hidden_dim)
        # cell: (num_layers * 2, batch_size, hidden_dim)
        lstm_out, (hidden, cell) = self.lstm(embeds)

        # 3. 取最后一个时间步的输出作为句子表示
        # 由于是双向 LSTM，最后一个时间步包含前向和后向的信息
        # 输出形状：(batch_size, hidden_dim * 2)
        out = lstm_out[:, -1, :]

        # 4. 应用 dropout
        out = self.dropout(out)

        # 5. 通过全连接层得到分类 logits
        # 输出形状：(batch_size, num_classes)
        out = self.fc(out)

        return out


# 示例用法
if __name__ == "__main__":
    # 假设词汇表大小为 1000，序列长度为 20
    vocab_size = 1000
    embed_dim = 128
    hidden_dim = 256
    num_classes = 2
    batch_size = 32
    seq_len = 20

    # 创建模型
    model = BiLSTM(vocab_size, embed_dim, hidden_dim, num_classes=num_classes)

    # 随机生成输入数据（词索引）
    inputs = torch.randint(0, vocab_size, (batch_size, seq_len))

    # 前向传播
    outputs = model(inputs)
    print("模型输出形状:", outputs.shape)  # 应为 (batch_size, num_classes)
